# CPC353 Assignment 1 ? Concordance Analysis

Run after `preprocessing.ipynb`.
This notebook generates keyword-in-context (KWIC) evidence from unfiltered sentence tokens, with original reviews and provenance retained. It uses the combined corpus, following the stop-word analysis convention. Current tracked inputs: the 65-review Shin Ramyun snapshot; rerun after any data refresh.

The analysis supports the question: which product attributes appear in English, Malay and code-switched instant noodle reviews, and how does surrounding language qualify them? It does not automatically classify sentiment or identify language switches.


## 1. Locate the project

Keep the repository files together in Jupyter/Colab; the helper uses only the Python standard library. No NLTK resource downloads are needed for this analysis.


In [1]:
from pathlib import Path
import sys

PROJECT_DIR = next((p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / "tools" / "concordance.py").is_file()
                    and (p / "data" / "processed").is_dir()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError("Open this notebook within the repository, with preprocessing outputs available.")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
from tools.concordance import run_analysis, concordance


## 2. Select terms and context width

Two explicit selection routes are combined: (1) the top 12 unfiltered token types after excluding project stop words and listed product-name tokens, and (2) a bilingual, research-question-led selection of attribute, evaluation, negation and contrast terms. Ties in frequency are alphabetical. Stop words are excluded only from candidate ranking, never from KWIC context. The curated selection is exploratory, not an exhaustive dictionary or a frequency-based attribute classifier.

Queries match exact case-folded token sequences; `taste` does not match `tastes`, and `pedas` does not match `pedasnya`. Add variants explicitly. Punctuation is already removed by preprocessing. Multiword expressions cannot cross estimated sentence boundaries. Overlapping occurrences are retained. Context is at most five tokens on each side within the same sentence. Read the full original for inter-sentential switches and punctuation.


In [2]:
WINDOW = 5
TOP_N = 12
BRAND_TOKENS = {"shin", "ramyun", "ramyum", "nongshim", "nongshim's", "maggi"}
QUERIES = [
    "taste", "flavour", "flavor", "rasa", "sedap", "spicy", "pedas", "salty", "masin",
    "soup", "broth", "kuah", "texture", "springy", "chewy", "kenyal", "lembik",
    "price", "harga", "mahal", "pricey", "packaging", "bungkusan", "packet", "portion",
    "not", "tak", "tidak", "but", "tapi", "sedap gila", "not too spicy",
]
result = run_analysis(PROJECT_DIR, QUERIES, window=WINDOW, top_n=TOP_N, excluded=BRAND_TOKENS)
manifest = result["manifest"]
print("Reviews:", manifest["review_count"])
print("Unfiltered tokens:", manifest["unfiltered_tokens"])
print("Frequency-selected terms:", ", ".join(manifest["frequency_candidates"]))
print("Distinct queries:", len(manifest["queries"]))
print("KWIC matches:", len(result["hits"]))


Reviews: 65
Unfiltered tokens: 1201
Frequency-selected terms: noodles, mi, but, kuah, sedap, kuahnya, broth, noodle, firm, kenyal, seasoning, tak
Distinct queries: 38
KWIC matches: 199


## 3. Keyword frequencies and review coverage

Occurrence count and distinct-review count answer different questions. Frequencies per 1,000 tokens use all unfiltered tokens as the denominator, including brand names and function words. Review identity is `(origin, review_id)`. A count within code-switched-labelled reviews does not prove the matched sentence itself switches languages. Queries with no hits stay in the summary.


In [3]:
print(f'{"Query":<18} {"Hits":>5} {"Reviews":>8} {"CS reviews":>11}')
for row in sorted(result["summary"], key=lambda r: (-r["occurrences"], r["query"])):
    print(f'{row["query"]:<18} {row["occurrences"]:>5} {row["reviews_containing"]:>8} {row["code_switched_reviews"]:>11}')


Query               Hits  Reviews  CS reviews
noodles               24       19           6
mi                    16       16           7
but                   13       13           6
kuah                  12       12           8
sedap                 12       11           7
kuahnya                9        9           3
broth                  8        7           1
noodle                 8        8           2
firm                   7        7           1
kenyal                 7        7           3
seasoning              7        6           1
tak                    7        6           5
flavour                5        5           3
masin                  5        5           1
packet                 5        5           2
soup                   5        5           2
spicy                  5        4           1
pedas                  4        4           3
springy                4        4           3
chewy                  3        3           1
lembik                 3        3 

## 4. Inspect context and original wording

The display below is a deterministic preview (first eight hits for each selected query); the CSV contains **every** match. It is not a representative sample. Change `INSPECT` to examine other terms or phrases. Offsets are zero-based review token positions, with an exclusive end, not original-text character offsets. Sentence text and the complete original review are exported with each hit.


In [4]:
INSPECT = ["not too spicy", "sedap gila", "tapi", "springy"]
for query in INSPECT:
    print("\nQuery:", query)
    matches = concordance(result["reviews"], [query], window=WINDOW)
    for h in matches[:8]:
        print(f'{h["origin"]}:{h["review_id"]} | {h["left_context"]} [{h["keyword"]}] {h["right_context"]}')
    print(f"Showing {min(8, len(matches))} of {len(matches)} matches")



Query: not too spicy
real:REV-06 | springy texture of the noodles [not too spicy] but the soup taste is
Showing 1 of 1 matches

Query: sedap gila
ai:REV-AI-034 | shin ramyun ni [sedap gila] bila mi masih firm dan
Showing 1 of 1 matches

Query: tapi
ai:REV-AI-019 | shin ramyun punya spice kuat [tapi] tambah sedikit air buat kuah
real:REV-09 |  [tapi] sedap jugak boleh ganti maggi
Showing 2 of 2 matches

Query: springy
ai:REV-AI-001 | ramyun memang pedas but the [springy] noodles make it worth cooking
ai:REV-AI-008 | mi shin ramyun ni [springy] kuah pun sedap sangat sampai
real:REV-06 | i like the [springy] texture of the noodles not
real:REV-12 | the noodle is [springy] and best serve with the
Showing 4 of 4 matches


## 5. Small bilingual lexicon and interpretation

`config/concordance_lexicon.csv` is a project-authored reading aid, not an automatic language tagger. Its entries retain informal forms and include glosses with interpretive cautions. `lexicon_evidence.csv` records actual occurrence counts and review IDs; `lexicon_kwic.csv` supplies full contexts. The glosses must be reviewed in context; `kaw` is explicitly treated as colloquial Malaysian usage rather than assigned a definitive lexical language.

Read `docs/concordance_results.md` for the current snapshot's discussion and explicitly marked AI examples. Recheck those conclusions after changing queries or inputs; the notebook regenerates evidence, not human interpretation.


In [5]:
for entry in result["lexicon"]:
    print(f'{entry["expression"]}: {entry["gloss"]} ? {entry["occurrences"]} occurrences')
print("\nSaved in data/analysis/concordance/:")
for name in ["kwic.csv", "keyword_summary.csv", "lexicon_evidence.csv", "lexicon_kwic.csv", "run_manifest.json"]:
    print(name)


sedap: tasty or delicious ? 12 occurrences
pedas: spicy ? 4 occurrences
kuah: broth or gravy ? 12 occurrences
kenyal: springy or chewy ? 7 occurrences
tak: not ? 7 occurrences
tapi: but ? 2 occurrences
sgt: sangat (very) ? 1 occurrences
mmg: memang (indeed or really) ? 1 occurrences
kaw: strong or concentrated ? 1 occurrences
sedap gila: very tasty ? 1 occurrences
not too spicy: moderate heat in context ? 1 occurrences
springy: elastic or resilient texture ? 4 occurrences

Saved in data/analysis/concordance/:
kwic.csv
keyword_summary.csv
lexicon_evidence.csv
lexicon_kwic.csv
run_manifest.json


## 6. Reliability and limits

The loader checks original-text SHA-256, unique origin/ID keys and agreement between flat and sentence tokens. Tests cover sentence/review boundaries, overlapping phrases, exact matching, negation, zero hits, candidate ranking and corrupt inputs. The manifest records input hashes, parameters and the helper hash. Output CSV files use UTF-8 with a BOM for Excel.

Sentence splitting is inherited from preprocessing and can oversplit abbreviations. Normalized contexts omit punctuation and do not replace originals. Frequency and keyword coverage alone cannot establish sentiment, attribute prevalence or switch rates. The current corpus is small, single-product and mostly synthetic; inherited language annotations and the flagged REV-01 source issue need human review. No token-level language labels are inferred. Additional products in the live Sheet or local collection folders are not automatically included.

Method reference: S. Bird, E. Klein and E. Loper, *Natural Language Processing with Python*, chapter 1, section 1.3, ?Searching Text?: https://www.nltk.org/book/ch01.html#searching-text. This project uses its own sentence-bounded KWIC routine to preserve IDs and provenance; it does not call `nltk.Text.concordance` on a concatenated corpus.
